In [34]:
from Bio import motifs
from Bio import SeqIO
from Bio.Seq import Seq
import numpy as np

In [23]:
# не понял откуда надо взять фасту поэтому взял человека
for record in SeqIO.parse("/Users/antonkorotkov/Homo_sapiens.GRCh38.dna.chromosome.1.fa", "fasta"):
    record = record[10000: 11000]
    print("ID:", record.id)
    print("Description:", record.description)
    print("Length:", len(record.seq))
    print("First 100 bp:", record.seq[:100])

ID: 1
Description: 1 dna:chromosome chromosome:GRCh38:1:1:248956422:1 REF
Length: 1000
First 100 bp: TAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAAC


In [24]:
nucl = ["A", "T", "G", "C"]
dinucs = [a + b for a in nucl for b in nucl]
counts = {x: 0 for x in dinucs}
record = record.seq
for i in range(len(record) - 1):
    pair = record[i: i + 2]
    if pair in counts:
        counts[pair] += 1

print(counts)  

{'AA': 86, 'AT': 4, 'AG': 57, 'AC': 95, 'TA': 75, 'TT': 2, 'TG': 20, 'TC': 15, 'GA': 35, 'GT': 18, 'GG': 55, 'GC': 112, 'CA': 46, 'CT': 87, 'CG': 88, 'CC': 204}


In [88]:
# посчитаем матрицу вероятностей. вот тут кипдим (точнее его отсутствие) меня жестко подставило, проблема в том что без
# него что 0 что 1 будет сумма по строкам
count_list = [v for k, v in counts.items()]
count_matrix = np.array(count_list).reshape(4, 4)
prob_matrix = count_matrix / np.sum(count_matrix, axis = 1, keepdims=True)
prob_matrix

array([[0.3553719 , 0.01652893, 0.23553719, 0.39256198],
       [0.66964286, 0.01785714, 0.17857143, 0.13392857],
       [0.15909091, 0.08181818, 0.25      , 0.50909091],
       [0.10823529, 0.20470588, 0.20705882, 0.48      ]])

In [89]:
prob_matrix.sum(axis=1)

array([1., 1., 1., 1.])

In [75]:
A = prob_matrix.T - np.eye(4) #матричное уравнение дает такую левую часть А ( A*pi^t = 0), но пи т - вырожденная, поэтому будет много доп решений
# поэтому добавим еще строчку из единиц (смысл в том что мы добавляем условие что сумма вероятностей будет 1)
A = np.vstack([A, np.ones(4)])
b = np.array([0,0,0,0,1])

pi, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

In [76]:
pi

array([0.24167976, 0.11135729, 0.22022595, 0.426737  ])

In [86]:
count = {x: 0 for x in nucl}
for s in record:
    count[s] += 1

print(np.array(list(count.values())) / np.sum(list(count.values())))

[0.242 0.112 0.22  0.426]


In [87]:
# ну совпадают да